# Can story attributes improve movie-rating prediction?
CS 7375 · Fall 2026 · Daryl Shy

This notebook prepares MovieLens ratings and CMU plot summaries for an experiment on story-attribute enrichment. The current cells define a provisional vocabulary, verify the raw datasets, and record input-file hashes. Matching, attribute extraction, and model evaluation have not yet been implemented.

## Research question
Do evidence-supported story attributes extracted from movie plot summaries improve prediction of users' original 1–5 ratings beyond user/movie identities and genres?

The planned experiment compares a factorization machine with three sets of inputs:

| Version | Inputs |
|---|---|
| A | User ID + movie ID |
| B | User ID + movie ID + MovieLens genres |
| C | User ID + movie ID + MovieLens genres + extracted story attributes |

**C versus B** measures the added value of story attributes. A provides an identity-only baseline. **RMSE** is the primary metric; **MAE** is secondary. All versions will use the same retained movies, ratings, and train/validation/test partitions, with comparable tuning. Held-out ratings will be reserved for evaluation.


## Data preparation and extraction design

MovieLens records will be matched to CMU summaries using **normalized title and exact release year**. Ambiguous matches will be excluded. Only movies with ratings and summaries meeting the provisional **100-word minimum** will enter the experiment.

The extraction pilot will use **20 summaries with three runs per summary** to evaluate supporting evidence, repeatability, and processing cost. The vocabulary and extraction procedure will be finalized before the full model comparison.

For each attribute:

- Require a meaningful role in the story and evidence from the supplied summary.
- Interpret the passage in context; a keyword alone is insufficient.
- Retain supporting passages and contextual notes for review.
- Use **not established by the summary** when evidence is insufficient; this does not prove the attribute is absent from the movie.
- Record extraction settings for reproducibility.

Formal attribute definitions, the extraction output schema, feature encoding, split proportions and seed, tuning settings, and treatment of unseen users or movies remain to be specified.


## Provisional story-attribute vocabulary

The following cell defines the 26 candidate attributes and the minimum summary length. It checks that the vocabulary contains exactly 26 distinct labels. Psychological change and bodily transformation are separate labels so that internal character development can be distinguished from physical transformation.


In [1]:
STORY_ATTRIBUTES = [
    "ambition/power", "revenge", "survival", "pursuit of freedom",
    "identity/self-discovery", "justice/injustice", "financial hardship",
    "friendship/alliance", "family conflict", "family reconciliation",
    "romantic connection", "romantic betrayal", "mentorship", "rivalry",
    "investigation", "deception/manipulation", "pursuit/escape", "rescue",
    "competition", "psychological change", "bodily transformation",
    "alternate realities/timelines", "addiction/recovery", "prejudice/exclusion",
    "caregiving/found family", "sacrifice/redemption",
]
MIN_SUMMARY_WORDS = 100  # Accepted provisional threshold; validate after matching.
assert len(STORY_ATTRIBUTES) == len(set(STORY_ATTRIBUTES)) == 26
print(f"{len(STORY_ATTRIBUTES)} provisional attributes; minimum {MIN_SUMMARY_WORDS} words.")

26 provisional attributes; minimum 100 words.


## Locate the input files

The following cell locates the project directory from the current working directory and builds paths to the required files:

- `MovieLens 1M Dataset/ml-1m/`: movie metadata, ratings, and user metadata.
- `CMU Movie Summary Corpus/MovieSummaries/`: movie metadata and plot summaries.

It checks that each required file exists before loading any data. Run the notebook from the project directory or its parent. Raw files are read without modification.


In [2]:
from pathlib import Path
from collections import Counter
import csv
import hashlib
import json

cwd = Path.cwd()
PROJECT_DIR = next((p for p in (cwd, cwd / "Semester Project", cwd.parent / "Semester Project")
                    if (p / "MovieLens 1M Dataset" / "ml-1m" / "ratings.dat").is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Open this notebook from the Semester Project folder or the AI course root.")
MOVIELENS_DIR = PROJECT_DIR / "MovieLens 1M Dataset" / "ml-1m"
CMU_DIR = PROJECT_DIR / "CMU Movie Summary Corpus" / "MovieSummaries"
required = [MOVIELENS_DIR / name for name in ("movies.dat", "ratings.dat", "users.dat")]
required += [CMU_DIR / name for name in ("movie.metadata.tsv", "plot_summaries.txt")]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(path)
print("Project:", PROJECT_DIR)
print("All required raw-data files are available.")

Project: /Users/darylshy/KSU/Fall 26/Artificial Intelligence/Semester Project
All required raw-data files are available.


## Load and validate MovieLens

MovieLens uses `::` separators; movie titles are read with Latin-1 encoding. The following cell loads movie IDs, titles, and genre lists, then counts ratings by movie, user, and rating value.

The assertions check unique movie and user identifiers, valid movie references, ratings within 1–5, and the expected dataset totals. The output distinguishes all movie metadata records from movies that actually received ratings.


In [3]:
# MovieLens files use :: separators. Titles are encoded in Latin-1.
ml_movies = {}
with (MOVIELENS_DIR / "movies.dat").open(encoding="latin-1") as handle:
    for line in handle:
        movie_id, title, genres = line.rstrip("\r\n").split("::")
        movie_id = int(movie_id)
        assert movie_id not in ml_movies
        ml_movies[movie_id] = {"title": title, "genres": genres.split("|")}

ratings_per_movie, ratings_per_user, rating_values = Counter(), Counter(), Counter()
with (MOVIELENS_DIR / "ratings.dat").open(encoding="ascii") as handle:
    for line in handle:
        user_id, movie_id, rating, timestamp = map(int, line.strip().split("::"))
        assert movie_id in ml_movies and 1 <= rating <= 5
        ratings_per_movie[movie_id] += 1
        ratings_per_user[user_id] += 1
        rating_values[rating] += 1

with (MOVIELENS_DIR / "users.dat").open(encoding="latin-1") as handle:
    user_ids = [int(line.split("::", 1)[0]) for line in handle]
assert len(user_ids) == len(set(user_ids))
assert set(ratings_per_user) <= set(user_ids)
assert sum(rating_values.values()) == 1_000_209
assert len(ml_movies) == 3_883 and len(ratings_per_movie) == 3_706
assert len(user_ids) == len(ratings_per_user) == 6_040
print(f"Movie metadata records: {len(ml_movies):,}")
print(f"Movies with ratings: {len(ratings_per_movie):,}")
print(f"Users: {len(user_ids):,}")
print(f"Ratings: {sum(rating_values.values()):,}")

Movie metadata records: 3,883
Movies with ratings: 3,706
Users: 6,040
Ratings: 1,000,209


## Load and validate CMU summaries

The following cell loads plot summaries and movie metadata into dictionaries keyed by Wikipedia movie ID. Each summary line is split at its first tab; metadata records contain nine tab-separated fields.

The assertions check record structure, unique identifiers, and expected totals. The final count shows how many summaries also have a movie metadata record, which is needed to obtain a title and release date for matching.


In [4]:
# CMU plot summaries and metadata are linked by Wikipedia movie ID.
cmu_plots = {}
with (CMU_DIR / "plot_summaries.txt").open(encoding="utf-8") as handle:
    for line in handle:
        wiki_id, summary = line.rstrip("\r\n").split("\t", 1)
        assert wiki_id not in cmu_plots
        cmu_plots[wiki_id] = summary

cmu_movies = {}
with (CMU_DIR / "movie.metadata.tsv").open(encoding="utf-8", newline="") as handle:
    for row in csv.reader(handle, delimiter="\t", quoting=csv.QUOTE_NONE):
        assert len(row) == 9
        wiki_id = row[0]
        assert wiki_id not in cmu_movies
        cmu_movies[wiki_id] = {"title": row[2], "release_date": row[3]}
assert len(cmu_plots) == 42_306
assert len(cmu_movies) == 81_741
print(f"CMU movie metadata records: {len(cmu_movies):,}")
print(f"CMU plot summaries: {len(cmu_plots):,}")
print(f"Summaries with movie metadata: {len(cmu_plots.keys() & cmu_movies.keys()):,}")

CMU movie metadata records: 81,741
CMU plot summaries: 42,306
Summaries with movie metadata: 42,207


## Record input-file fingerprints

The following cell calculates a SHA-256 hash for each required input file. It reads files in chunks and prints their project-relative paths with their hashes. These fingerprints identify the exact raw files used and allow subsequent runs to detect changes in the inputs.


In [5]:
# Record hashes in notebook output to identify the exact input files.
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

input_fingerprints = {str(p.relative_to(PROJECT_DIR)): sha256_file(p) for p in required}
print(json.dumps(input_fingerprints, indent=2))

{
  "MovieLens 1M Dataset/ml-1m/movies.dat": "0140fc2356357c1a851d0f52e893a1e4d3696df632c4141cea8d5bc3d621f0b9",
  "MovieLens 1M Dataset/ml-1m/ratings.dat": "506d64ca44484487c11dc2d9a28de5c54948213e6b96285e298afe28d6ea4e0f",
  "MovieLens 1M Dataset/ml-1m/users.dat": "1dc3a95300cb19f7c10e027daf29b8cdf1d908dab65b5edfb45950aa389a10a3",
  "CMU Movie Summary Corpus/MovieSummaries/movie.metadata.tsv": "6152760aa98c0c261a3eec50dff73293e93b7e899e4be07ffeb86c27ddecb0f5",
  "CMU Movie Summary Corpus/MovieSummaries/plot_summaries.txt": "718ab6d359fbae81b8f7d224fb4d9f3d8fc1010b495ba458baaf7936b29d680d"
}


## References

- Abdollahpouri et al. (2021). *User-centered Evaluation of Popularity Bias in Recommender Systems*. UMAP, 119–129. [DOI](https://doi.org/10.1145/3450613.3456821). Local copy: `Evaluation of Popularity Bias/ALS.pdf`. This paper supplies recommender-system background; its recommendation-list metrics are distinct from the rating-error metrics used here.
- Harper and Konstan (2015). *The MovieLens Datasets: History and Context*. The original README is included with the local MovieLens files.
- Bamman, O'Connor, and Smith (2013). *Learning Latent Personas of Film Characters*. ACL. [CMU Movie Summary Corpus](https://www.cs.cmu.edu/~ark/personas/). Preserve the corpus README and Creative Commons Attribution-ShareAlike attribution.

CMU archive SHA-256: `6c3bd3d42046c4b87e2267f2e2599ba6f651b8ad23a8998237c3d2bb338ac559`.
